# CORTEX.COMPLETE — LLM Completion Patterns

`SNOWFLAKE.CORTEX.COMPLETE` is the foundational function for calling LLMs inside Snowflake SQL.

| Item | Detail |
|---|---|
| **Function** | `SNOWFLAKE.CORTEX.COMPLETE` |
| **Models** | `claude-3-5-haiku`, `mistral-large2`, `llama3.1-70b` |
| **Use-case** | Prompt engineering patterns for production SQL workflows |


## Step 1 — Environment Setup

In [ ]:
CREATE DATABASE IF NOT EXISTS GENAI_LEARNING;
CREATE SCHEMA IF NOT EXISTS GENAI_LEARNING.PUBLIC;
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Basic Completion

The simplest pattern: model name + prompt string.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'claude-3-5-haiku',
    'Explain data warehousing in exactly three sentences.'
) AS response;

## Step 3 — System + User Message Pattern

Use the structured message format for role-based prompting.

In [ ]:
SELECT SNOWFLAKE.CORTEX.COMPLETE(
    'claude-3-5-haiku',
    [
        {'role': 'system', 'content': 'You are a concise Snowflake expert. Answer in bullet points.'},
        {'role': 'user', 'content': 'What are the key benefits of Snowflake Cortex AI functions?'}
    ],
    {'temperature': 0.3, 'max_tokens': 500}
) AS response;

## Step 4 — Applying LLM to Table Data

In [ ]:
CREATE OR REPLACE TABLE prompt_experiments (
    experiment_id INT AUTOINCREMENT,
    pattern_name  VARCHAR(100),
    prompt_text   TEXT
);

INSERT INTO prompt_experiments (pattern_name, prompt_text) VALUES
('zero_shot', 'Classify this text as POSITIVE, NEGATIVE, or NEUTRAL: The new feature update broke my workflow completely.'),
('few_shot', 'Classify sentiment. Examples:\nInput: Love it! -> POSITIVE\nInput: Terrible. -> NEGATIVE\nInput: It is okay. -> NEUTRAL\n\nInput: The new feature update broke my workflow completely. ->'),
('chain_of_thought', 'Analyze the sentiment step by step:\n1. Identify key phrases\n2. Determine emotional tone\n3. Give final classification\n\nText: The new feature update broke my workflow completely.'),
('extraction', 'Extract the following from this support ticket in JSON format: {issue_type, severity, product_area}.\nTicket: The dashboard export feature has been timing out for the past 3 days. This is blocking our weekly executive report.'),
('summarization', 'Summarize the key technical decisions in this meeting note in exactly 3 bullet points:\nWe discussed migrating from PostgreSQL to Snowflake. Team agreed on a phased approach starting with analytics workloads. Data engineering will own the pipeline migration while the BI team handles dashboard conversion. Target completion is Q2.');

In [ ]:
SELECT
    pattern_name,
    SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-haiku', prompt_text) AS llm_response
FROM prompt_experiments
ORDER BY experiment_id;

## Step 5 — Compare Models

In [ ]:
SELECT
    'claude-3-5-haiku' AS model,
    SNOWFLAKE.CORTEX.COMPLETE('claude-3-5-haiku', 'In one sentence, what makes Snowflake unique?') AS response
UNION ALL
SELECT
    'mistral-large2',
    SNOWFLAKE.CORTEX.COMPLETE('mistral-large2', 'In one sentence, what makes Snowflake unique?')
UNION ALL
SELECT
    'llama3.1-70b',
    SNOWFLAKE.CORTEX.COMPLETE('llama3.1-70b', 'In one sentence, what makes Snowflake unique?');

## Key Takeaways

- `COMPLETE` supports both simple string prompts and structured message arrays.
- Use `temperature` (0.0-1.0) to control creativity; lower = more deterministic.
- Five key patterns: zero-shot, few-shot, chain-of-thought, extraction, summarization.
- Apply to table columns with SELECT for batch processing at scale.
- Compare models on your specific task — accuracy and latency vary by use-case.
